# PetGuard AI | Final Application and Evaluation

**ITAI 1378 Computer Vision and AI**

Process fixed-camera video to detect dogs and cats, check a selected restricted zone, and export annotated video and violation records.

**Workflow:** A. Setup and processing | B. Export run | C. Prepare labels | D. Upload reviewed labels | E1–E4. Validate, evaluate, review, and download.

**Evaluation note:** Model predictions are not ground truth. Labels reviewed only from annotated output remain provisional until checked against original footage. Results from one short clip do not establish general accuracy. Colab runtime files are temporary, so download outputs before disconnecting.

## A1. Set Up the Project

Run once per new Colab runtime to clone the repository and install dependencies. An existing checkout is preserved and does not automatically receive newer commits.


In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path('/content/PetGuard-AI')
URL = 'https://github.com/iamhuongthn/PetGuard-AI.git'
if not (REPO / '.git').is_dir():
    if REPO.exists():
        raise RuntimeError(f'{REPO} exists but is not a Git clone; inspect it before proceeding.')
    subprocess.run(['git','clone','--branch','master',URL,str(REPO)],check=True)
else:
    print('Existing repository retained; no destructive reset performed.')
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')],check=True)
assert (REPO/'src/app.py').is_file()
print('Repository ready:',REPO)

## A2. Process a Video

Launch Gradio, upload a fixed-camera MP4, select the zone type, draw at least three corners, and confirm the polygon. Click **Process Video** to generate annotated output and event records.

A violation is confirmed after five consecutive positive frames at the scene level. Individual pets are not tracked across frames. The temporary Gradio share link is not a permanent demo link.

In [ ]:
import sys
sys.path.insert(0,str(REPO))
from src.app import build_app
app = build_app()
app.queue(default_concurrency_limit=1).launch(share=True, show_error=True, prevent_thread_lock=True)

## B. Save the Completed Video Run

Export the latest processing run as a ZIP containing annotated videos, `events.csv`, `frame_predictions.csv`, and `zone_config.json`. Each run has a unique ID. Download the ZIP before the Colab runtime disconnects.

In [ ]:
from google.colab import files
from scripts.export_run import export
runs_dir = REPO/'results'/'runs'
runs = sorted((p for p in runs_dir.iterdir() if p.is_dir() and (p/'run_metadata.json').is_file()), key=lambda p:p.stat().st_mtime) if runs_dir.exists() else []
assert runs, 'No runs found. Process a video in Gradio first.'
latest = runs[-1]
required = ['frame_predictions.csv','zone_config.json','events.csv']
assert all((latest/x).is_file() for x in required), f'Run {latest.name} is incomplete.'
run_zip = export(latest, f'/content/petguard_run_{latest.name}.zip')
print('Exported run:',latest.name)
files.download(str(run_zip))

## C. Prepare a blind review pack

Upload the original MP4 used for the selected run. Review the sampled frames with only the zone outline, without opening predictions.


In [ ]:
from google.colab import files
from scripts.review_frames import prepare
import shutil
RUN_DIR = latest
uploaded = files.upload()
assert len(uploaded)==1
original = Path(next(iter(uploaded))).resolve()
pack = prepare(RUN_DIR, original, Path('/content')/f'review_{RUN_DIR.name}')
files.download(shutil.make_archive(str(pack), 'zip', pack))


## D. Upload reviewed labels

Complete the blind CSV with actual pet counts, SAFE or VIOLATION, reviewer and review date. Predictions are joined after review.


In [ ]:
from google.colab import files
from io import BytesIO
import pandas as pd
uploaded = files.upload()
assert len(uploaded)==1
labels = pd.read_csv(BytesIO(next(iter(uploaded.values()))))
assert 'predicted_zone_status' not in labels
assert labels.video_id.eq(RUN_DIR.name).all()
verified_path = REPO/'data'/f'zone_labels_reviewed_{RUN_DIR.name}.csv'
labels.to_csv(verified_path,index=False)


## E1. Validate Labels

Check required columns, unique IDs, pet counts, and valid `SAFE` or `VIOLATION` values. Review label counts before evaluation. Structural validation does not verify label correctness.

In [ ]:
df = pd.read_csv(verified_path)
assert not df.empty and df.sample_id.is_unique and df.frame_id.is_unique
assert df.expected_zone_status.isin(['SAFE','VIOLATION']).all()
assert df[['reviewer','review_date','source_sha256']].notna().all().all()
counts = pd.to_numeric(df.pet_count_ground_truth,errors='raise')
assert ((counts>=0)&(counts%1==0)).all()
print(df.expected_zone_status.value_counts())


## E2. Generate Evaluation Results

Compare expected and predicted frame-level zone statuses. Save accuracy, false violation rate, violation recall, confusion matrix, failure cases, and a summary in `results/metrics/`. These metrics do not measure confirmed-event accuracy.

In [ ]:
from scripts.evaluate_zone_labels import evaluate
metrics_dir = REPO/'results'/'metrics'/f'metrics_{RUN_DIR.name}_reviewed'
evaluate(verified_path, metrics_dir, RUN_DIR)


## E3. Review Results

Inspect the metrics, confusion matrix, and missed violations. Compare zone accuracy with the 90% target and false violation rate with the 10% maximum. Report limitations and keep annotated-video labels provisional until independently checked.



In [ ]:
from IPython.display import display, Markdown
display(Markdown((metrics_dir/'zone_evaluation_summary.md').read_text()))
display(pd.read_csv(metrics_dir/'zone_failure_cases.csv'))


## E4. Download evaluation files

Extract the ZIP into the repository root. Existing provisional reports remain separate.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
zip_path = Path('/content')/f'petguard_reviewed_{RUN_DIR.name}.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    z.write(verified_path,verified_path.relative_to(REPO))
    for f in metrics_dir.iterdir():
        if f.is_file(): z.write(f,f.relative_to(REPO))
files.download(str(zip_path))
